# Capítulo 8 — Medir la semejanza · Notebook para completar

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch08_a_completer.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Las celdas que debes modificar llevan la indicación **PARA COMPLETAR**. Ejecuta las celdas en orden: mientras falte una respuesta, las comprobaciones lo indican sin error. La solución está en `ch08_corrige.ipynb`, en la misma página complementaria.

**Índice.** Datos · Los cálculos de las lecciones 8.1 a 8.3 · Ejercicios 8.1 a 8.6

**Antes de abrir Jupyter:** los ejercicios de este capítulo se hacen primero en papel, con una calculadora. Las celdas de abajo sirven para **comprobar** tus cálculos. Como en el libro, las medias y las desviaciones típicas dividen por $n$ y se calculan sobre las seis ventas de referencia, A a F.

**Nombres de las variables.** El código usa los nombres en inglés del libro: `floor_area` (superficie, en m²), `distance` (distancia al centro, en km), `price` (precio, en k€), `home` (nombre de la vivienda).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: viviendas de referencia en azul, vecinos elegidos rodeados de naranja, distancias en magenta
BLUE, ORANGE, MAGENTA = '#2a78d6', '#eb6834', '#c2185b'

## Datos

Los identificadores D2.1 y D2.E2 son los del libro y de los archivos CSV. Las columnas se renombran en inglés al leerlas; la correspondencia figura en un comentario.

In [ ]:
# Datos del capítulo 8, copiados de los archivos CSV del libro (no modificar)

# D2.1 — las siete viviendas; columnas del CSV renombradas en inglés:
#   home = logement, floor_area = surface_m2 (m²), distance = distance_km (km), price = prix_kEUR (k€)
homes = pd.DataFrame({
    'home':       ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area': [32, 45, 54, 70, 85, 98, 62],
    'distance':   [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':      [118, 172, 185, 245, 279, 335, 209],
})
reference = homes[homes['home'] != 'G'].reset_index(drop=True)   # las seis ventas de referencia, A a F
X = reference[['floor_area', 'distance']].to_numpy(dtype=float)  # una fila por venta: (m² ; km)
price = reference['price'].to_numpy(dtype=float)                 # precios de las ventas de referencia, en k€
G = np.array([62, 2.5])                                          # la vivienda G: 62 m², 2,5 km (vendida por 209 k€)

# D2.E2 — la vivienda I (ejercicio 8.2)
I = np.array([75, 3.5])   # 75 m², 3,5 km

# Ejemplos del libro (lecciones 8.1 y 8.3): plano de la ciudad en km, mensajes, embeddings inventados
school, G_map = np.array([4.5, 6]), np.array([1.5, 2])            # la escuela y G en el plano, en km
R1, R2, P = np.array([3, 1]), np.array([6, 2]), np.array([1, 3])  # («reunión», «oferta»)
embeddings = {'reunión': np.array([4, 3]), 'asamblea': np.array([3, 4]),
              'rebajas': np.array([-3, 4]), 'promoción': np.array([-4, 3])}

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `home` | identificador de la vivienda (A, B…) | — | `logement` |
| `floor_area` | superficie habitable de la vivienda | m² | `surface_m2` |
| `price` | precio de venta | k€ | `prix_kEUR` |
| `distance` | distancia al centro de la ciudad | km | `distance_km` |
| `distance_m` | distancia al centro de la ciudad | m | — |
| `message` | identificador del texto (T1 a T4) | — | — |
| `meeting_count` | número de apariciones de la palabra «reunión» | apariciones | — |
| `offer_count` | número de apariciones de la palabra «oferta» | apariciones | — |
| `volunteer_count` | número de apariciones de la palabra «voluntario» | apariciones | — |

In [ ]:
# Herramientas proporcionadas: no tienes que modificarlas

def nearest(dists, k=3):
    """Las k ventas de referencia más cercanas (distancias dadas en el orden A a F) y su precio medio."""
    order = np.argsort(dists, kind='stable')[:k]
    return list(reference['home'].iloc[order]), float(price[order].mean())


def show_table(dists, label):
    """Tabla de las distancias a una vivienda, de la más corta a la más larga."""
    t = pd.DataFrame({'home': reference['home'], label: np.round(dists, 3), 'price': price})
    print(t.sort_values(label, kind='stable').to_string(index=False))

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 8.1 a 8.3. Ejecútalas y compara con el libro.

In [ ]:
# Lección 8.1 — norma y distancia (ecuación 8.1)
trip = school - G_map                       # desplazamiento de G a la escuela, en km
print(f"desplazamiento {trip}, distancia {np.sqrt((trip ** 2).sum()):g} km")
print(f"del centro a G: {np.linalg.norm(G_map):g} km")             # la distancia al centro de la tabla 2.1
print(f"norma de (2 ; 3 ; 6): {np.linalg.norm([2, 3, 6]):g}")      # Pitágoras dos veces, o un solo cálculo
C_home = X[2]                                                       # C: 54 m², 2,5 km
print(f"distancia de C a G: {np.linalg.norm(C_home - G):g} (unidades mezcladas: m² y km)")

In [ ]:
# Lección 8.2 — primer intento: distancias brutas a G, después distancia al centro en metros
raw_G = np.sqrt(((X - G) ** 2).sum(axis=1))
show_table(raw_G, 'raw_distance')
n_km, e_km = nearest(raw_G)
print(f"3 vecinos más cercanos (bruto, km): {n_km}, estimación {e_km:.1f} k€")
X_m, G_m = X * [1, 1000], G * [1, 1000]                             # distancia al centro en metros
n_m, e_m = nearest(np.linalg.norm(X_m - G_m, axis=1))
print(f"3 vecinos más cercanos (bruto, m) : {n_m}, estimación {e_m:.0f} k€")

In [ ]:
# Lección 8.2 — segundo intento: estandarizar con las ventas de referencia (ecuación 8.2)
mu, sigma = X.mean(axis=0), X.std(axis=0)                           # división por n (ddof=0)
print(f"superficie: media {mu[0]:.0f} m², desviación típica {sigma[0]:.2f} m²")
print(f"distancia: media {mu[1]:.3f} km, desviación típica {sigma[1]:.3f} km (√29/6 = {np.sqrt(29) / 6:.3f})")
Z, z_G = (X - mu) / sigma, (G - mu) / sigma
print('z de G:', np.round(z_G, 2))
std_G = np.linalg.norm(Z - z_G, axis=1)
show_table(std_G, 'std_distance')
neighbours, estimate = nearest(std_G)
print(f"3 vecinos más cercanos (estandarizado): {neighbours}, estimación {estimate:.0f} k€ (precio real: 209 k€)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(Z[:, 0], Z[:, 1], color=BLUE, s=40)
ax.scatter(*z_G, marker='D', color='black', s=40)
for h, (a, b) in zip(reference['home'], Z):
    ax.annotate(h, (a, b), xytext=(5, 4), textcoords='offset points')
ax.annotate('G', z_G, xytext=(5, 4), textcoords='offset points')
idx = np.argsort(std_G, kind='stable')[:3]
ax.scatter(Z[idx, 0], Z[idx, 1], s=220, facecolors='none', edgecolors=ORANGE, linewidths=1.8)
ax.add_patch(plt.Circle(z_G, std_G[idx[-1]], fill=False, linestyle='--', color='grey'))
ax.set_aspect('equal'); ax.set_xlabel('z superficie'); ax.set_ylabel('z distancia'); ax.grid(alpha=0.4)
plt.show()

In [ ]:
# Lección 8.3 — ¿distancia o coseno? (ecuación 8.3)
def cos_lesson(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"distancias: R1–R2 {np.linalg.norm(R1 - R2):.2f}, R1–P {np.linalg.norm(R1 - P):.2f}")
print(f"cosenos   : R1–R2 {cos_lesson(R1, R2):.2f}, R1–P {cos_lesson(R1, P):.2f}, "
      f"ángulo R1–P {np.degrees(np.arccos(cos_lesson(R1, P))):.0f}°")

# Vista previa: los embeddings de la tabla 8.3 (todos de norma 5)
for a, b in [('reunión', 'asamblea'), ('rebajas', 'promoción'), ('reunión', 'rebajas'), ('reunión', 'promoción')]:
    print(f"cos({a}, {b}) = {cos_lesson(embeddings[a], embeddings[b]):.2f}")
message = (embeddings['reunión'] + embeddings['asamblea']) / 2       # el mensaje «reunión, asamblea»
print(f"mensaje {message}: cos con reunión {cos_lesson(message, embeddings['reunión']):.2f}, "
      f"con promoción {cos_lesson(message, embeddings['promoción']):.2f}")

## Ejercicio 8.1 — Distancias en el plano de la ciudad

*Nociones: norma y distancia (lección 8.1, ecuación 8.1).* En el plano de la figura 8.1, en kilómetros, la piscina está en el punto $(7{,}5 ; 10)$ y la estación en el punto $(-3{,}5 ; 5)$; G está en el punto $(1{,}5 ; 2)$.

### a) a e) En papel

*Tus cálculos:* …

In [ ]:
# PARA COMPLETAR: la norma de un vector, sin np.linalg.norm (ecuación 8.1)
def norm(u):
    u = np.asarray(u, dtype=float)
    return None   # PARA COMPLETAR: raíz cuadrada de la suma de los cuadrados de las coordenadas

pool, station = np.array([7.5, 10]), np.array([-3.5, 5])   # piscina y estación, en km

In [ ]:
# Comprobación de a) a e)
if norm([6, 8]) is None:
    print('aún sin completar. Completa la función norm.')
else:
    d_pool, d_station, d_station_pool = norm(pool - G_map), norm(station - G_map), norm(pool - station)
    print(f"a) {norm([6, 8]):g}   b) {d_pool:g} km   c) {d_station:.2f} km y {d_station_pool:.2f} km")
    print(f"d) rodeo: {d_station + d_station_pool - d_pool:.2f} km   e) {norm([2, 3, 6]):g}")
    print('→', 'correcto' if abs(d_pool - 10) < 1e-9 and abs(d_station - 5.831) < 5e-4
          and abs(d_station_pool - 12.083) < 5e-4 and abs(norm([2, 3, 6]) - 7) < 1e-9 else 'revisar')

## Ejercicio 8.2 — Los vecinos de la vivienda I

*Nociones: distancia bruta y distancia estandarizada (lección 8.2, ecuación 8.2).* La vivienda I (conjunto D2.E2) tiene un `floor_area` de 75 m² y una `distance` de 3,5 km: es el vector `I`. La estimamos con las seis ventas de referencia y $k = 3$.

- `floor_area` — superficie de la vivienda, en m² (columna `surface_m2` del CSV);
- `distance` — distancia al centro, en km (columna `distance_km` del CSV);
- `price` — precio de las ventas de referencia, en k€; puntuaciones $z$ sin unidad.

### a) a e) En papel

*Tus cálculos y tus interpretaciones:* …

In [ ]:
# PARA COMPLETAR: distancias de I a las seis ventas A a F, brutas y después estandarizadas
raw_I = None   # PARA COMPLETAR: np.sqrt(((X - I) ** 2).sum(axis=1))
z_I = None     # PARA COMPLETAR: puntuaciones de I con mu y sigma (lección 8.2)
std_I = None   # PARA COMPLETAR: distancias entre las filas de Z y z_I

In [ ]:
# Comprobación de a) a c)
if raw_I is None or z_I is None or std_I is None:
    print('aún sin completar. Completa las tres líneas.')
else:
    show_table(raw_I, 'raw_distance')
    (n_raw, e_raw), (n_std, e_std) = nearest(raw_I), nearest(std_I)
    print(f"a) bruto: {n_raw}, estimación {e_raw:.1f} k€")
    print(f"b) z de I: {np.round(z_I, 2)}")
    show_table(std_I, 'std_distance')
    print(f"c) estandarizado: {n_std}, estimación {e_std:.0f} k€")
    print('→', 'correcto' if n_raw == ['D', 'E', 'C'] and abs(e_raw - 236.33) < 0.01
          and n_std == ['E', 'C', 'A'] and abs(e_std - 194) < 1e-9 else 'revisar')

## Ejercicio 8.3 — ¿Metros o kilómetros?

*Nociones: distancia bruta y unidades (lección 8.2).* Retomamos G y las seis ventas, pero la distancia al centro se mide en metros: `distance_m` $= 1\,000 \times$ `distance`.

- `distance_m` — distancia al centro, en m;
- `distance` — distancia al centro, en km; `floor_area` (m²) y `price` (k€) de las ventas de referencia.

### a) a d) En papel

*Tus cálculos y tus respuestas:* …

In [ ]:
# PARA COMPLETAR: la distancia al centro en metros, después las distancias brutas a G
distance_m = None     # PARA COMPLETAR: 1000 × la columna distance de las ventas A a F
G_distance_m = None   # PARA COMPLETAR: distancia de G al centro, en m
raw_G_m = None        # PARA COMPLETAR: distancias brutas a G con (floor_area ; distance_m)

In [ ]:
# Comprobación de a), c) y d)
if distance_m is None or G_distance_m is None or raw_G_m is None:
    print('aún sin completar. Completa las tres líneas.')
else:
    show_table(raw_G_m, 'raw_distance_m')
    n_m, e_m = nearest(raw_G_m)
    print(f"a) en metros: {n_m}, estimación {e_m:.0f} k€")
    mu_m, sigma_m = distance_m.mean(), distance_m.std()
    z_G_m = (G_distance_m - mu_m) / sigma_m
    print(f"c) media {mu_m:.0f} m, desviación típica {sigma_m:.0f} m; z de G: {z_G_m:.2f} en m, {z_G[1]:.2f} en km")
    print('→', 'correcto' if n_m == ['C', 'E', 'A'] and abs(e_m - 194) < 1e-9 and abs(z_G_m - z_G[1]) < 1e-12
          else 'revisar')

## Ejercicio 8.4 — ¿Cuántos vecinos?

*Nociones: vecinos más cercanos, elección de $k$ (lección 8.2), validación (lección 6.4).* En el orden de las distancias estandarizadas a G, las seis ventas y su `price` son C (185 k€), E (279), B (172), A (118), D (245) y F (335).

- `price` — precio de venta, en k€; $k$ — número de vecinos elegidos, sin unidad.

### a) a c) En papel

*Tus estimaciones y tu respuesta:* …

In [ ]:
# PARA COMPLETAR: la estimación de G para cada valor de k
estimates = None   # PARA COMPLETAR: {k: nearest(std_G, k)[1] for k in (1, 2, 3, 4, 6)}

In [ ]:
# Comprobación de a) y c)
if estimates is None:
    print('aún sin completar. Completa la línea.')
else:
    for k, e in estimates.items():
        print(f"k = {k}: {e:.1f} k€, diferencia con el precio real (209 k€): {e - 209:+.1f} k€")
    expected = {1: 185, 2: 232, 3: 212, 4: 188.5, 6: 1334 / 6}
    print('→', 'correcto' if all(abs(estimates[k] - v) < 1e-9 for k, v in expected.items()) else 'revisar')

## Ejercicio 8.5 — ¿Distancia o coseno?

*Nociones: distancia (ecuación 8.1), coseno (ecuación 8.3).* Contamos tres palabras en cuatro mensajes de la asociación: «reunión», «oferta» y «voluntario» (tabla 8.4 del libro).

- `message` — identificador del mensaje, T1 a T4;
- `meeting_count`, `offer_count`, `volunteer_count` — número de usos de las palabras «reunión», «oferta» y «voluntario» en el mensaje: las tres coordenadas de su vector.

In [ ]:
# Datos del ejercicio 8.5 (tabla del libro)
words = pd.DataFrame({
    'message':         ['T1', 'T2', 'T3', 'T4'],
    'meeting_count':   [2, 6, 1, 0],
    'offer_count':     [0, 0, 2, 1],
    'volunteer_count': [1, 3, 2, 0],
}).set_index('message')
T = {m: words.loc[m].to_numpy(dtype=float) for m in words.index}
print(words)

### a) a d) En papel

*Tus cálculos y tu interpretación:* …

In [ ]:
# PARA COMPLETAR: el coseno de dos vectores (ecuación 8.3)
def cosine(u, v):
    return None   # PARA COMPLETAR: producto escalar dividido por el producto de las normas

In [ ]:
# Comprobación de a) a c)
if cosine(T['T1'], T['T2']) is None:
    print('aún sin completar. Completa la función cosine.')
else:
    results = pd.DataFrame({
        'norm': [np.linalg.norm(T[m]) for m in T],
        'distance_to_T1': [np.linalg.norm(T['T1'] - T[m]) for m in T],
        'cosine_with_T1': [cosine(T['T1'], T[m]) for m in T],
    }, index=list(T)).round(2)
    print(results)
    print('→', 'correcto' if abs(cosine(T['T1'], T['T2']) - 1) < 1e-12 and abs(cosine(T['T1'], T['T3']) - 0.596) < 5e-4
          and abs(cosine(T['T1'], T['T4'])) < 1e-12 else 'revisar')

## Ejercicio 8.6 — Reto: normalizar y después comparar

*Nociones: norma y producto escalar (ecuación 8.1), coseno (ecuación 8.3).* Las preguntas a) y b) son demostraciones en papel; el notebook comprueba c) y d).

### a) a d) En papel

*Tus demostraciones y tus cálculos:* …

In [ ]:
# PARA COMPLETAR: cada mensaje dividido por su norma, después las distancias a T1 normalizado
normalized = None     # PARA COMPLETAR: {m: T[m] / np.linalg.norm(T[m]) for m in T}
unit_dist = None      # PARA COMPLETAR: {m: distancia entre normalized['T1'] y normalized[m] for m in T}

In [ ]:
# Comprobación de c) y d)
if normalized is None or unit_dist is None:
    print('aún sin completar. Completa las dos líneas.')
else:
    for m in ('T2', 'T3', 'T4'):
        c = float(np.dot(T['T1'], T[m]) / (np.linalg.norm(T['T1']) * np.linalg.norm(T[m])))
        print(f"{m}: distancia normalizada {unit_dist[m]:.2f}, √(2 − 2 cos) = {np.sqrt(max(2 - 2 * c, 0)):.2f}")
    u, v = embeddings['reunión'] / 5, embeddings['asamblea'] / 5
    print(f"d) reunión y asamblea: directamente {np.linalg.norm(u - v):.2f}, con b) {np.sqrt(2 - 2 * np.dot(u, v)):.2f}")
    print('→', 'correcto' if abs(unit_dist['T2']) < 1e-9 and abs(unit_dist['T3'] - 0.899) < 5e-4
          and abs(unit_dist['T4'] - np.sqrt(2)) < 1e-9 else 'revisar')